In [2]:
import polars as pl
import sqlalchemy as sa
from IPython.display import display

DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = "password"
DB_CONECTION = "athena"

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)

NOMBRES = ["Carlos", "Maria", "Juan", "Ana", "Luis", "Elena", "Pedro", "Lucia", "Miguel", "Sofia", "Jorge", "Laura"]

with engine.begin() as conn:
    carreras = conn.execute(sa.text("""
        SELECT id_carrera, nombre_carrera
        FROM carrera
        ORDER BY id_carrera;
    """)).mappings().all()

    for carrera in carreras:
        id_carrera = carrera["id_carrera"]
        correo = f"estudiante_carrera_{id_carrera}@athena.edu"
        nombre = NOMBRES[(id_carrera - 1) % len(NOMBRES)]

        usuario = conn.execute(sa.text("""
            SELECT id_usuario, uuid_usuario
            FROM usuario
            WHERE correo = :correo;
        """), {"correo": correo}).mappings().first()

        if usuario is None:
            usuario = conn.execute(sa.text("""
                INSERT INTO usuario (nombre, correo)
                VALUES (:nombre, :correo)
                RETURNING id_usuario, uuid_usuario;
            """), {"nombre": nombre, "correo": correo}).mappings().one()
        else:
            conn.execute(sa.text("""
                UPDATE usuario
                SET nombre = :nombre
                WHERE id_usuario = :id_usuario;
            """), {"nombre": nombre, "id_usuario": usuario["id_usuario"]})

        id_usuario = usuario["id_usuario"]
        uuid_usuario = usuario["uuid_usuario"]

        habilidades = conn.execute(sa.text("""
            SELECT id_habilidad
            FROM carrera_habilidad
            WHERE id_carrera = :id_carrera
            ORDER BY id_habilidad;
        """), {"id_carrera": id_carrera}).mappings().all()

        conn.execute(sa.text("""
            DELETE FROM usuario_habilidad
            WHERE id_usuario = :id_usuario;
        """), {"id_usuario": id_usuario})

        if habilidades:
            conn.execute(sa.text("""
                INSERT INTO usuario_habilidad
                    (id_usuario, id_habilidad, cumplimiento_criterio)
                VALUES (:id_usuario, :id_habilidad, 100.0);
            """), [
                {"id_usuario": id_usuario, "id_habilidad": habilidad["id_habilidad"]}
                for habilidad in habilidades
            ])

        conn.execute(sa.text("""
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, 100.0, 1)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        """), {"uuid_usuario": uuid_usuario, "id_carrera": id_carrera})

    query_resultados = """
        SELECT
            u.id_usuario,
            u.nombre AS nombre_usuario,
            c.id_carrera,
            c.nombre_carrera AS carrera,
            r.porcentaje_coincidencia,
            h.id_habilidad,
            h.descripcion AS habilidad,
            uh.cumplimiento_criterio
        FROM resultado r
        JOIN usuario u ON u.uuid_usuario = r.uuid_usuario
        JOIN carrera c ON c.id_carrera = r.id_carrera
        JOIN carrera_habilidad ch ON ch.id_carrera = c.id_carrera
        JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
        JOIN usuario_habilidad uh
            ON uh.id_usuario = u.id_usuario
            AND uh.id_habilidad = h.id_habilidad
        ORDER BY c.id_carrera, u.id_usuario, h.id_habilidad;
    """

    resultados_df = pl.read_database(query_resultados, connection=conn)

print(f"Carreras procesadas: {len(carreras)}")
print(f"Filas de habilidades/resultados: {resultados_df.height}")
display(resultados_df)

ModuleNotFoundError: No module named 'psycopg2'